In [1]:
# ===== Import libraries =====
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

# ===== Load dataset =====
url = "https://drive.google.com/file/d/1JPFEjSOlQ-gWExGiUv2kB1-yHWGfZdOQ/view?usp=sharing"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]

# Load dataset
housing = pd.read_csv(path)
housing.head()

# ===== Define features and target =====
X = housing.copy()
y = np.log1p(X.pop("SalePrice"))   # log-transform target for Kaggle metric

# ===== Split data into train and test sets =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== Separate categorical and numerical columns =====
X_cat = X_train.select_dtypes(include=["object"])
X_num = X_train.select_dtypes(exclude=["object"])

# ===== Build preprocessing pipeline =====
preprocessor = make_column_transformer(
    (
        make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        ),
        X_cat.columns
    ),
    (
        SimpleImputer(strategy="median"),
        X_num.columns
    ),
    remainder="drop"
)

# ===== Build full pipeline =====
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression)),
    ("model", LinearRegression())
])

# ===== Define parameter grid =====
param_grid = {
    "feature_selection__k": [5, 10, 15, 20, 25, 30, "all"]
}

# ===== Build GridSearchCV =====
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",   # competition metric
    n_jobs=-1
)

# ===== Train GridSearchCV =====
grid.fit(X_train, y_train)

# ===== Show best parameter and best cross-validation score =====
print("Best k:", grid.best_params_["feature_selection__k"])
print("Best CV RMSE (log scale):", -grid.best_score_)

# ===== Use the best model =====
best_model = grid.best_estimator_

# ===== Predict on the test set =====
y_pred = best_model.predict(X_test)

# ===== Evaluate the best model on the test set =====
print("Test MAE (log scale):", mean_absolute_error(y_test, y_pred))
print("Test RMSE (log scale):", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test MAPE (log scale):", mean_absolute_percentage_error(y_test, y_pred))
print("Test R2 (log scale):", r2_score(y_test, y_pred))

# ===== Optional: convert predictions back to original SalePrice scale =====
y_test_original = np.expm1(y_test)
y_pred_original = np.expm1(y_pred)

print("Test RMSE (original SalePrice scale):", mean_squared_error(y_test_original, y_pred_original) ** 0.5)

# ===== Show selected feature names =====
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
selected_mask = best_model.named_steps["feature_selection"].get_support()
selected_features = pd.Series(feature_names[selected_mask], name="Selected Features")

print("\nSelected features:")
print(selected_features.to_string(index=False))

Best k: 30
Best CV RMSE (log scale): 0.16312627561970566
Test MAE (log scale): 0.11701763216373534
Test RMSE (log scale): 0.1598323156431153
Test MAPE (log scale): 0.009858555486004542
Test R2 (log scale): 0.8631036182970951
Test RMSE (original SalePrice scale): 29484.616394146058

Selected features:
       pipeline__MasVnrType
        pipeline__ExterQual
       pipeline__Foundation
         pipeline__BsmtQual
     pipeline__BsmtExposure
        pipeline__HeatingQC
       pipeline__CentralAir
      pipeline__KitchenQual
       pipeline__GarageType
     pipeline__GarageFinish
       pipeline__PavedDrive
 simpleimputer__LotFrontage
 simpleimputer__OverallQual
   simpleimputer__YearBuilt
simpleimputer__YearRemodAdd
  simpleimputer__MasVnrArea
  simpleimputer__BsmtFinSF1
 simpleimputer__TotalBsmtSF
    simpleimputer__1stFlrSF
    simpleimputer__2ndFlrSF
   simpleimputer__GrLivArea
    simpleimputer__FullBath
    simpleimputer__HalfBath
simpleimputer__TotRmsAbvGrd
  simpleimputer__Fireplace

#COMPETITION-Kaggle

In [2]:
# ===== Load teacher's external test data =====
url = "https://drive.google.com/file/d/1q14sdW_8Gk9x0h5fAejPpq38xuRwgHnY/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]

# ===== Load teacher test CSV =====

test_df = pd.read_csv(path)

In [5]:
# ===== Use test data WITHOUT dropping Id =====
X_teacher_test = test_df.copy()

# ===== Predict =====
test_pred_log = best_model.predict(X_teacher_test)

# ===== Convert back =====
test_pred = np.expm1(test_pred_log)

# ===== Build submission =====
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred
})

# ===== Save =====
submission.to_csv("submission_SKB_LR.csv", index=False)

submission.head()

,Id,SalePrice
0,1461,117173.629367
1,1462,149547.799299
2,1463,174126.856607
3,1464,196828.475251
4,1465,191790.224950


In [8]:
from google.colab import files
files.download("submission_SKB_LR.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>